In [ ]:
from transformers import (
    MarianMTModel, MarianTokenizer, pipeline
)
import torch
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm


## . FINBERT

FinBERT is a pre-trained NLP model to analyze sentiment of financial text. It is built by further training the BERT language model in the finance domain, using a large financial corpus and thereby fine-tuning it for financial sentiment classification. [https://huggingface.co/ProsusAI/finbert]<br>

This pre-trained model was trained in english text, knowing that we have non-english tweets we will test with three differnt scenarios:

- With the raw data;
- With the translated data;
- Only with the english data

In [ ]:
def evaluate_model_predictions(y_pred_train, y_pred_val, y_train, y_val, show_confusion_matrix = True, show_classification_report = True):
    """
    Evaluate the performance of a classification model on training and validation sets.

    Computes and prints accuracy, F1-score (macro), precision, and recall for both sets.
    Optionally displays the confusion matrix and detailed classification report for the validation set.

    Parameters:
    y_pred_train (array-like): Predicted labels for the training set.
    y_pred_val (array-like): Predicted labels for the validation set.
    y_train (array-like, optional): True labels for the training set.
    y_val (array-like, optional): True labels for the validation set.
    show_confusion_matrix (bool): If True, prints the confusion matrix for the validation set.
    show_classification_report (bool): If True, prints the classification report for the validation set.

    Returns:
    tuple: A tuple containing:
        - train_accuracy (float)
        - train_f1 (float)
        - val_accuracy (float)
        - val_f1 (float)
    """

    if y_train is None:
        y_train = y_train_raw
    if y_val is None:
        y_val = y_val_raw
    
    # to get the accurancy and the macro f1 score of the model on the training set
    train_accuracy = accuracy_score(y_train, y_pred_train)
    train_f1 = f1_score(y_train, y_pred_train, average='macro')
    
    # to get the accurancy and the macro f1 score of the model on the validation set
    val_accuracy = accuracy_score(y_val, y_pred_val)
    val_f1 = f1_score(y_val, y_pred_val, average='macro')

    # to get the precision and the recall of the model on the validation set
    val_precision = precision_score(y_val, y_pred_val, average='macro')
    val_recall = recall_score(y_val, y_pred_val, average='macro')

    print(f"Accuracy of train: {train_accuracy:.4f}")
    print(f"F1 Macro (Train): {train_f1:.4f}")
    print(f"Accuracy of val: {val_accuracy:.4f}")
    print(f"\033[1mF1 Macro (Val)\033[0m: {val_f1:.4f}")
    print(f"Precision (Val): {val_precision:.4f}")
    print(f"Recall (Val): {val_recall:.4f}")
    
    # to get the confusion matrix and the classification report of the model on the validation set
    if show_confusion_matrix==True:
        print('\nConfusion Matrix for Validation Data:')    
        print(confusion_matrix(y_val, y_pred_val))

    if show_classification_report==True:
        print('\nClassification Report for Validation Data:')
        print(classification_report(y_val, y_pred_val))

    return val_accuracy, val_f1, val_precision, val_recall

In [ ]:
MODEL = "ProsusAI/finbert"

# Load Hugging Face pipeline
hf_classifier = pipeline(
    "text-classification",
    model=MODEL,
    tokenizer=MODEL,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### .1 With the raw data

In [ ]:

# Make predictions
preds_train = hf_classifier(list(X_train_raw))
preds_val = hf_classifier(list(X_val_raw))

# Map model's labels to integers
label_to_int = {
    "positive": 1,  # Bullish
    "negative": 0,  # Bearish
    "neutral":  2   # Neutral
}

y_pred_train = [label_to_int[pred["label"]] for pred in preds_train]
y_pred_val   = [label_to_int[pred["label"]] for pred in preds_val]

# Evaluation
finberta_accuracy, finberta_f1_macro, finberta_precision, finberta_recall = evaluate_model_predictions(y_pred_train=y_pred_train, y_pred_val=y_pred_val, y_train = y_train_raw, y_val = y_val_raw)

Accuracy of train: 0.7117
F1 Macro (Train): 0.6619
Accuracy of val: 0.7140
F1 Macro (Val): 0.6627
Precision (Val): 0.6441
Recall (Val): 0.7059

Confusion Matrix for Validation Data:
[[230  16  42]
 [ 50 225 110]
 [172 156 908]]

Classification Report for Validation Data:
              precision    recall  f1-score   support

           0       0.51      0.80      0.62       288
           1       0.57      0.58      0.58       385
           2       0.86      0.73      0.79      1236

    accuracy                           0.71      1909
   macro avg       0.64      0.71      0.66      1909
weighted avg       0.75      0.71      0.72      1909



### .2 With the translated data

To tranlate the non-English tweets to English Helsinki-NLP/opus-mt-{src_lang}-en famaly of models was used. Which for each  src_lang will tranlate to english. [https://huggingface.co/Helsinki-NLP]

In [ ]:
SUPPORTED_LANGS = ['en', 'fr', 'es', 'de', 'it', 'nl', 'ru', 'zh', 'ja', 'ar', 'fi', 'hu', 'vi', 'sv', 'cs', 'da', 'is']
loaded_models = {}

def load_translation_model(src_lang):
    model_name = f'Helsinki-NLP/opus-mt-{src_lang}-en'
    if model_name not in loaded_models:
        tokenizer = MarianTokenizer.from_pretrained(model_name)
        model = MarianMTModel.from_pretrained(model_name)
        loaded_models[model_name] = (tokenizer, model)
    return loaded_models[model_name]

def translate_text(text, src_lang):
    if src_lang == 'en' or src_lang not in SUPPORTED_LANGS:
        return text  
    try:
        tokenizer, model = load_translation_model(src_lang)
        batch = tokenizer([text], return_tensors="pt", padding=True, truncation=True)
        generated = model.generate(**batch)
        translated = tokenizer.decode(generated[0], skip_special_tokens=True)
        return translated
    except Exception as e:
        print(f"Error translating ({src_lang}): {e}")
        return text

train['translated_text'] = [
    translate_text(text, lang)
    for text, lang in tqdm(zip(train['text'], train['language']), total=len(train))]


# to see it its translating well:
train[train['language'] == 'fr']

  0%|          | 0/9543 [00:00<?, ?it/s]/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

 15%|█▍        | 1415/9543 [00:03<00:22, 367.70it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

 92%|█████████▏| 8790/9543 [01:29<00:20, 36.48it/s] 

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

100%|██████████| 9543/9543 [01:34<00:00, 101.22it/s]


,text,label,sentiment,clean_text,clean_text_no_sw,light_clean,language,translated_text
1449,Eudonet annonce un chiffre d’affaires récurren...,2,Neutral,eudonet annonce un chiffre d affaire récurrent...,eudonet annonce un chiffre affaire récurrent e...,eudonet annonce un chiffre d’affaires récurren...,fr,Eudonet announces a 32% increase in recurring ...
1837,Rio Tinto’s Giant Mongolia Project Dealt Anoth...,2,Neutral,rio tinto s giant mongolia project dealt anoth...,rio tinto giant mongolia project dealt another...,rio tinto’s giant mongolia project dealt anoth...,fr,Rio Tinto的s Giant Mongolia Project Dealt Anoth...
1962,Tinyclues accélère sa croissance en 2019,2,Neutral,tinyclues accélère sa croissance en,tinyclues accélère sa croissance en,tinyclues accélère sa croissance en 2019,fr,Tinyclues accelerates its growth in 2019
4318,Cloudia lance la dernière génération de soluti...,2,Neutral,cloudia lance la dernière génération de soluti...,cloudia lance la dernière génération de soluti...,cloudia lance la dernière génération de soluti...,fr,Cloudia launches the latest generation of Proc...
4785,Korian s’associe à Omedys pour développer le p...,2,Neutral,korian s associe à omedys pour développer le p...,korian associe à omedys pour développer le pre...,korian s’associe à omedys pour développer le p...,fr,Korian is partnering with Omedys to develop th...
5352,Total : Résultats du quatrième trimestre et de...,2,Neutral,total résultats du quatrième trimestre et de l...,total résultats du quatrième trimestre et de l...,total : résultats du quatrième trimestre et de...,fr,Total: Results for the fourth quarter and the ...
5499,Will China Declare Force Majeure If Virus Situ...,2,Neutral,will china declare force majeure if virus situ...,china declare force majeure virus situation wo...,will china declare force majeure if virus situ...,fr,Will China Declare Force Major If Virus Situat...
8448,DFND,2,Neutral,dfnd,dfnd,dfnd,fr,DFND
8590,HOML,2,Neutral,homl,homl,homl,fr,HOML
8707,Is Les Hôtels de Paris (EPA:HDP) Using Too Muc...,2,Neutral,is le hôtels de paris epa hdp using too much d...,le hôtels de paris epa hdp using much debt?,is les hôtels de paris (epa:hdp) using too muc...,fr,Is Hotels in Paris (EPA:HDP) Using Too Much Debt?


In [ ]:
X_val_translated = train.loc[X_val_raw.index, 'translated_text']
X_train_translated = train.loc[X_train_raw.index, 'translated_text']
# Make predictions
preds_train = hf_classifier(list(X_train_translated))
preds_val = hf_classifier(list(X_val_translated))

# Map model's labels to integers
label_to_int = {
    "positive": 1,  # Bullish
    "negative": 0,  # Bearish
    "neutral":  2   # Neutral
}

y_pred_train = [label_to_int[pred["label"]] for pred in preds_train]
y_pred_val   = [label_to_int[pred["label"]] for pred in preds_val]

# Evaluation
finberta_accuracy_trans, finberta_f1_macro_trans, finberta_precision_trans, finberta_recall_trans = evaluate_model_predictions(y_pred_train=y_pred_train, y_pred_val=y_pred_val, y_train = y_train_raw, y_val = y_val_raw)

Accuracy of train: 0.7109
F1 Macro (Train): 0.6613
Accuracy of val: 0.7114
F1 Macro (Val): 0.6606
Precision (Val): 0.6415
Recall (Val): 0.7045

Confusion Matrix for Validation Data:
[[230  16  42]
 [ 50 225 110]
 [172 161 903]]

Classification Report for Validation Data:
              precision    recall  f1-score   support

           0       0.51      0.80      0.62       288
           1       0.56      0.58      0.57       385
           2       0.86      0.73      0.79      1236

    accuracy                           0.71      1909
   macro avg       0.64      0.70      0.66      1909
weighted avg       0.74      0.71      0.72      1909



### .3 With only the english data

In [ ]:
eng_mask_val = train.loc[X_val_raw.index, 'language'] == 'en'
y_val_eng = y_val_raw.loc[eng_mask_val[eng_mask_val].index]
X_val_eng = X_val_raw.loc[eng_mask_val[eng_mask_val].index]

eng_mask_train = train.loc[X_train_raw.index, 'language'] == 'en'
y_train_eng = y_train_raw.loc[eng_mask_train[eng_mask_train].index]
X_train_eng = X_train_raw.loc[eng_mask_train[eng_mask_train].index]

# Make predictions
preds_train = hf_classifier(list(X_train_eng))
preds_val = hf_classifier(list(X_val_eng))

# Map model's labels to integers
label_to_int = {
    "positive": 1,  # Bullish
    "negative": 0,  # Bearish
    "neutral":  2   # Neutral
}

y_pred_train = [label_to_int[pred["label"]] for pred in preds_train]
y_pred_val   = [label_to_int[pred["label"]] for pred in preds_val]

# Evaluation
finberta_accuracy_trans, finberta_f1_macro_trans, finberta_precision_trans, finberta_recall_trans = evaluate_model_predictions(y_pred_train=y_pred_train, y_pred_val=y_pred_val, y_train = y_train_eng, y_val = y_val_eng)


Accuracy of train: 0.7098
F1 Macro (Train): 0.6615
Accuracy of val: 0.7125
F1 Macro (Val): 0.6625
Precision (Val): 0.6440
Recall (Val): 0.7055

Confusion Matrix for Validation Data:
[[230  16  42]
 [ 50 225 109]
 [171 156 893]]

Classification Report for Validation Data:
              precision    recall  f1-score   support

           0       0.51      0.80      0.62       288
           1       0.57      0.59      0.58       384
           2       0.86      0.73      0.79      1220

    accuracy                           0.71      1892
   macro avg       0.64      0.71      0.66      1892
weighted avg       0.74      0.71      0.72      1892



### Conclusion

| Type of data | Tran F1 score | Validation F1score | 
|---|---|---|
|  raw data | 0.6619 |  0.6627 | 
| translated data | 0.6613 | 0.6606| 
| only english data | 0.6615 | 0.6625 | 

As we can see there is not much difference between the three different data, this can be due the fake positives of the model when predicting the language of each tweet. This can happen, for example, if an English tweet is talking about a foreigner company the model may pick on that and assume that the tweet is the language of the foreigner company. Also, for the translated data it can add noise if we translate the data wrongly classified with a foreigner langue or lose the contextual meaning during translation. FLAG VER SE TA BEM ESCRITO

The best data was the original one, the raw data, where we got the results of the F1 score 0.6619 and 0.6627, for train and validation respectfully.

## . BERTtweet

BERTweet shares the same architecture as BERT-base, but it’s pretrained like RoBERTa on English Tweets. It performs really well on Tweet-related tasks like part-of-speech tagging, named entity recognition, and text classification. [https://huggingface.co/docs/transformers/model_doc/bertweet]

Since our goal is to do sentimental analyse of the stock tweets we find this model intresting t know how it will performe with our data.<br>
Like FINBERT, this model was trained as well in english tweets so we will preside as well with three differnt scenarios:

- With the raw data;
- With the translated data;
- Only with the english data

In [ ]:
# Choose a pretrained BERTweet sentiment classifier
MODEL = "finiteautomata/bertweet-base-sentiment-analysis"

# Load Hugging Face pipeline for text classification
hf_classifier = pipeline(
    "text-classification",
    model=MODEL,
    tokenizer=MODEL,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True
)


### .1 With raw data

In [ ]:

# Make predictions on raw validation tweets
preds_train = hf_classifier(list(X_train_raw))
preds_val = hf_classifier(list(X_val_raw))

# Map the model’s label strings to your integer classes
label_to_int = {
    "NEG": 0,  # negative → Bearish
    "NEU": 2,  # neutral  → Neutral
    "POS": 1   # positive → Bullish
}

y_pred_train = [label_to_int[pred['label']] for pred in preds_train]
y_pred_val = [label_to_int[pred['label']] for pred in preds_val]


# Evaluation
bertweet_accuracy, bertweet_f1_macro, bertweet_precision, bertweet_recall = evaluate_model_predictions(y_pred_train=y_pred_train, y_pred_val=y_pred_val, y_train = y_train_raw, y_val = y_val_raw)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Accuracy of train: 0.6719
F1 Macro (Train): 0.5893
Accuracy of val: 0.6679
F1 Macro (Val): 0.5911
Precision (Val): 0.5937
Recall (Val): 0.5928

Confusion Matrix for Validation Data:
[[179   2 107]
 [ 11 151 223]
 [131 160 945]]

Classification Report for Validation Data:
              precision    recall  f1-score   support

           0       0.56      0.62      0.59       288
           1       0.48      0.39      0.43       385
           2       0.74      0.76      0.75      1236

    accuracy                           0.67      1909
   macro avg       0.59      0.59      0.59      1909
weighted avg       0.66      0.67      0.66      1909



### With translated data

In [ ]:
X_val_translated = train.loc[X_val_raw.index, 'translated_text']
X_train_translated = train.loc[X_train_raw.index, 'translated_text']
# Load Hugging Face pipeline for text classification
hf_classifier = pipeline(
    "text-classification",
    model=MODEL,
    tokenizer=MODEL,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True
)

# Make predictions on raw validation tweets
preds_train = hf_classifier(list(X_train_translated))
preds_val = hf_classifier(list(X_val_translated))

# Map the model’s label strings to your integer classes
label_to_int = {
    "NEG": 0,  # negative → Bearish
    "NEU": 2,  # neutral  → Neutral
    "POS": 1   # positive → Bullish
}

y_pred_train = [label_to_int[pred['label']] for pred in preds_train]
y_pred_val = [label_to_int[pred['label']] for pred in preds_val]


# Evaluation
bertweet_accuracy, bertweet_f1_macro, bertweet_precision, bertweet_recall = evaluate_model_predictions(y_pred_train=y_pred_train, y_pred_val=y_pred_val, y_train = y_train_raw, y_val = y_val_raw)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Accuracy of train: 0.6702
F1 Macro (Train): 0.5881
Accuracy of val: 0.6653
F1 Macro (Val): 0.5892
Precision (Val): 0.5909
Recall (Val): 0.5914

Confusion Matrix for Validation Data:
[[179   2 107]
 [ 11 151 223]
 [131 165 940]]

Classification Report for Validation Data:
              precision    recall  f1-score   support

           0       0.56      0.62      0.59       288
           1       0.47      0.39      0.43       385
           2       0.74      0.76      0.75      1236

    accuracy                           0.67      1909
   macro avg       0.59      0.59      0.59      1909
weighted avg       0.66      0.67      0.66      1909



### With only english data

In [ ]:
eng_mask_val = train.loc[X_val_raw.index, 'language'] == 'en'
y_val_eng = y_val_raw.loc[eng_mask_val[eng_mask_val].index]
X_val_eng = X_val_raw.loc[eng_mask_val[eng_mask_val].index]

eng_mask_train = train.loc[X_train_raw.index, 'language'] == 'en'
y_train_eng = y_train_raw.loc[eng_mask_train[eng_mask_train].index]
X_train_eng = X_train_raw.loc[eng_mask_train[eng_mask_train].index]

# Make predictions
preds_train = hf_classifier(list(X_train_eng))
preds_val = hf_classifier(list(X_val_eng))

# Map model's labels to integers
label_to_int = {
    "NEG": 0,  # negative → Bearish
    "NEU": 2,  # neutral  → Neutral
    "POS": 1   # positive → Bullish
}


y_pred_train = [label_to_int[pred["label"]] for pred in preds_train]
y_pred_val   = [label_to_int[pred["label"]] for pred in preds_val]

# Evaluation
bertweet_accuracy, bertweet_f1_macro, bertweet_precision, bertweet_recall = evaluate_model_predictions(y_pred_train=y_pred_train, y_pred_val=y_pred_val, y_train = y_train_eng, y_val = y_val_eng)

Accuracy of train: 0.6695
F1 Macro (Train): 0.5886
Accuracy of val: 0.6649
F1 Macro (Val): 0.5895
Precision (Val): 0.5921
Recall (Val): 0.5912

Confusion Matrix for Validation Data:
[[179   2 107]
 [ 11 150 223]
 [131 160 929]]

Classification Report for Validation Data:
              precision    recall  f1-score   support

           0       0.56      0.62      0.59       288
           1       0.48      0.39      0.43       384
           2       0.74      0.76      0.75      1220

    accuracy                           0.66      1892
   macro avg       0.59      0.59      0.59      1892
weighted avg       0.66      0.66      0.66      1892



### Conclusion

| Type of data | Tran F1 score | Validation F1score | 
|---|---|---|
|  raw data | 0.5893 |  0.5911 | 
| translated data | 0.5881 | 0.5892| 
| only english data | 0.5886 | 0.5895 | 

Similiar to FINBERT, BERTtweet produces consistent F1 scores throughout the three scenarios. The raw data got better results, probably due the same reasons as in FINBERT, the non-English tweets don't add noise, but trying to translate or remove them may add noise and remove important keys to classify the tweets.